In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT = Path("/home/bannanakid/RCRL_on_RCMDP")
sys.path.insert(0, str(PROJECT))

from envs.half_cheetah_jointspeed_cost import HalfCheetahJointSpeedCost
from configs.halfcheetah_safety_thresholds import JOINT_SPEED_THRESHOLDS, JOINT_SPEED_SCALES, COST_VERSION

JOINT_NAMES = (
    "back_thigh",
    "back_shin",
    "back_foot",
    "front_thigh",
    "front_shin",
    "front_foot",
)

RUN = 1
FIRST_SEED = 1005
EPISODES = 50
EVALUATE_FREQ = 25

UNCONSTRAINED_ACTOR_PATH = PROJECT / "models" / "HalfCheetahUnconstrained" / f"run{RUN}" / "Best_Unconstrained_actor"
JOINT_ACTOR_PATH = PROJECT / "models" / "HalfCheetahJointSpeedCost" / f"run{RUN}" / "Best_JointSpeedCost_actor"
OUTPUT = PROJECT / "plot_data" / "HalfCheetahJointSpeedCost" / f"run{RUN}" / "evaluation"
OUTPUT.mkdir(parents=True, exist_ok=True)

joint_thresholds = np.asarray(JOINT_SPEED_THRESHOLDS, dtype=np.float64)
joint_scales = np.asarray(JOINT_SPEED_SCALES, dtype=np.float64)

print("Project:", PROJECT)
print("Joint-speed thresholds:", joint_thresholds)
print("Joint-speed scales:", joint_scales)
print("Cost version:", COST_VERSION)
print("Held-out seeds:", FIRST_SEED, "through", FIRST_SEED + EPISODES - 1)
print("Unconstrained actor:", UNCONSTRAINED_ACTOR_PATH)
print("Joint-speed actor:", JOINT_ACTOR_PATH)

In [ ]:
if JOINT_SPEED_THRESHOLDS is None or JOINT_SPEED_SCALES is None:
    raise RuntimeError("JOINT_SPEED_THRESHOLDS and JOINT_SPEED_SCALES must be finalized before evaluation")

if joint_thresholds.shape != (6,) or joint_scales.shape != (6,):
    raise RuntimeError("Joint-speed thresholds and scales must each contain six values")

if np.any(joint_scales <= 0):
    raise RuntimeError("Joint-speed scales must all be positive")

if not UNCONSTRAINED_ACTOR_PATH.exists():
    raise FileNotFoundError(UNCONSTRAINED_ACTOR_PATH)

if not JOINT_ACTOR_PATH.exists():
    raise FileNotFoundError(
        f"{JOINT_ACTOR_PATH}\nNo feasible Best_JointSpeedCost_actor was found. Check the joint-speed training evaluation outputs."
    )

print("Both actor files found")

In [ ]:
class ActorGaussian(nn.Module):
    def __init__(self, state_dim, hidden_width, action_dim, max_action):
        super().__init__()
        self.max_action = float(max_action)
        self.fc1 = nn.Linear(state_dim, hidden_width)
        self.fc2 = nn.Linear(hidden_width, hidden_width)
        self.fc3 = nn.Linear(hidden_width, hidden_width)
        self.mean_layer = nn.Linear(hidden_width, action_dim)
        self.log_std = nn.Parameter(torch.zeros(1, action_dim))
        self.activate_func = nn.ReLU()

    def forward(self, state):
        state = self.activate_func(self.fc1(state))
        state = self.activate_func(self.fc2(state))
        state = self.activate_func(self.fc3(state))
        return self.max_action * torch.tanh(self.mean_layer(state))


def unwrap_reset(reset_result):
    result = reset_result
    while isinstance(result, tuple):
        result = result[0]
    return np.asarray(result, dtype=np.float64)


def load_actor(path, env):
    state = torch.load(path, map_location="cpu")

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    if isinstance(state, dict) and "actor_state_dict" in state:
        state = state["actor_state_dict"]

    actor = ActorGaussian(
        int(state["fc1.weight"].shape[1]),
        int(state["fc1.weight"].shape[0]),
        int(state["mean_layer.weight"].shape[0]),
        float(env.action_space.high[0]),
    )

    actor.load_state_dict(state)
    actor.eval()
    return actor

In [ ]:
probe_env = HalfCheetahJointSpeedCost()

unconstrained_actor = load_actor(UNCONSTRAINED_ACTOR_PATH, probe_env)
joint_actor = load_actor(JOINT_ACTOR_PATH, probe_env)

print("Observation dimension:", probe_env.observation_space.shape[0])
print("Action dimension:", probe_env.action_space.shape[0])
print("Unconstrained parameters:", sum(p.numel() for p in unconstrained_actor.parameters()))
print("Joint-speed parameters:", sum(p.numel() for p in joint_actor.parameters()))

In [ ]:
def collect_policy(actor, label, first_seed, episodes):
    env = HalfCheetahJointSpeedCost()

    episode = []
    reset_seed = []
    step = []
    reward = []
    cost = []
    xpos = []
    joint_velocities = []
    joint_speeds = []
    joint_excess = []
    joint_normalized_excess = []
    maximum_joint_speed = []
    fastest_joint_index = []
    actions = []
    maximum_action_abs = []

    episode_rewards = []
    episode_costs = []
    episode_max_costs = []
    episode_lengths = []
    episode_net_displacements = []
    episode_absolute_displacements = []
    episode_traveled_distances = []

    for ep in range(episodes):
        ep_seed = first_seed + ep
        obs = unwrap_reset(env.reset(seed=ep_seed))

        start_x = float(env.sim.data.qpos[0])
        previous_x = start_x

        total_reward = 0.0
        total_cost = 0.0
        max_cost = 0.0
        traveled = 0.0
        t = 0
        done = False

        while not done:
            x = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)

            with torch.no_grad():
                action = actor(x).cpu().numpy().reshape(-1)

            action = np.clip(action, env.action_space.low, env.action_space.high)
            obs_next, r, c, truncated, terminated, info = env.step(action)
            done = bool(truncated or terminated)

            current_x = float(info.get("xpos", env.sim.data.qpos[0]))
            velocities = np.asarray(
                info.get("joint_velocities", env.sim.data.qvel[3:9]),
                dtype=np.float64,
            ).copy()
            speeds = np.asarray(
                info.get("joint_speeds", np.abs(velocities)),
                dtype=np.float64,
            ).copy()
            excess = np.maximum(speeds - joint_thresholds, 0.0)
            normalized_excess = excess / joint_scales
            fastest = int(np.argmax(speeds))

            traveled += abs(current_x - previous_x)
            previous_x = current_x

            episode.append(ep + 1)
            reset_seed.append(ep_seed)
            step.append(t)
            reward.append(float(r))
            cost.append(float(c))
            xpos.append(current_x)
            joint_velocities.append(velocities)
            joint_speeds.append(speeds)
            joint_excess.append(excess)
            joint_normalized_excess.append(normalized_excess)
            maximum_joint_speed.append(float(np.max(speeds)))
            fastest_joint_index.append(fastest)
            actions.append(np.asarray(action, dtype=np.float64).copy())
            maximum_action_abs.append(float(np.max(np.abs(action))))

            total_reward += float(r)
            total_cost += float(c)
            max_cost = max(max_cost, float(c))

            obs = obs_next
            t += 1

        end_x = float(env.sim.data.qpos[0])

        episode_rewards.append(total_reward)
        episode_costs.append(total_cost)
        episode_max_costs.append(max_cost)
        episode_lengths.append(t)
        episode_net_displacements.append(end_x - start_x)
        episode_absolute_displacements.append(abs(end_x - start_x))
        episode_traveled_distances.append(traveled)

        print(
            f"{label}: episode {ep + 1}/{episodes} seed={ep_seed} "
            f"reward={total_reward:.3f} total_cost={total_cost:.6f} max_cost={max_cost:.6f}"
        )

    return {
        "label": label,
        "episode": np.asarray(episode, dtype=np.int64),
        "reset_seed": np.asarray(reset_seed, dtype=np.int64),
        "step": np.asarray(step, dtype=np.int64),
        "reward": np.asarray(reward, dtype=np.float64),
        "cost": np.asarray(cost, dtype=np.float64),
        "xpos": np.asarray(xpos, dtype=np.float64),
        "joint_velocities": np.asarray(joint_velocities, dtype=np.float64),
        "joint_speeds": np.asarray(joint_speeds, dtype=np.float64),
        "joint_excess": np.asarray(joint_excess, dtype=np.float64),
        "joint_normalized_excess": np.asarray(joint_normalized_excess, dtype=np.float64),
        "maximum_joint_speed": np.asarray(maximum_joint_speed, dtype=np.float64),
        "fastest_joint_index": np.asarray(fastest_joint_index, dtype=np.int64),
        "actions": np.asarray(actions, dtype=np.float64),
        "maximum_action_abs": np.asarray(maximum_action_abs, dtype=np.float64),
        "episode_rewards": np.asarray(episode_rewards, dtype=np.float64),
        "episode_costs": np.asarray(episode_costs, dtype=np.float64),
        "episode_max_costs": np.asarray(episode_max_costs, dtype=np.float64),
        "episode_lengths": np.asarray(episode_lengths, dtype=np.int64),
        "episode_net_displacements": np.asarray(episode_net_displacements, dtype=np.float64),
        "episode_absolute_displacements": np.asarray(episode_absolute_displacements, dtype=np.float64),
        "episode_traveled_distances": np.asarray(episode_traveled_distances, dtype=np.float64),
    }

In [ ]:
unconstrained = collect_policy(
    unconstrained_actor,
    "Unconstrained",
    FIRST_SEED,
    EPISODES,
)

joint_constrained = collect_policy(
    joint_actor,
    "Joint-speed constrained",
    FIRST_SEED,
    EPISODES,
)

In [ ]:
def save_dataset(data, path):
    np.savez_compressed(
        path,
        episode=data["episode"],
        reset_seed=data["reset_seed"],
        step=data["step"],
        reward=data["reward"],
        cost=data["cost"],
        xpos=data["xpos"],
        joint_velocities=data["joint_velocities"],
        joint_speeds=data["joint_speeds"],
        joint_excess=data["joint_excess"],
        joint_normalized_excess=data["joint_normalized_excess"],
        maximum_joint_speed=data["maximum_joint_speed"],
        fastest_joint_index=data["fastest_joint_index"],
        actions=data["actions"],
        maximum_action_abs=data["maximum_action_abs"],
        episode_rewards=data["episode_rewards"],
        episode_costs=data["episode_costs"],
        episode_max_costs=data["episode_max_costs"],
        episode_lengths=data["episode_lengths"],
        episode_net_displacements=data["episode_net_displacements"],
        episode_absolute_displacements=data["episode_absolute_displacements"],
        episode_traveled_distances=data["episode_traveled_distances"],
        joint_names=np.asarray(JOINT_NAMES),
        joint_speed_thresholds=joint_thresholds,
        joint_speed_scales=joint_scales,
        cost_version=np.asarray(str(COST_VERSION)),
        first_seed=np.asarray(FIRST_SEED, dtype=np.int64),
        number_of_episodes=np.asarray(EPISODES, dtype=np.int64),
    )


save_dataset(
    unconstrained,
    OUTPUT / "unconstrained_on_joint_speed_env.npz",
)

save_dataset(
    joint_constrained,
    OUTPUT / "joint_speed_constrained_evaluation.npz",
)

print("Saved:", OUTPUT / "unconstrained_on_joint_speed_env.npz")
print("Saved:", OUTPUT / "joint_speed_constrained_evaluation.npz")

In [ ]:
def longest_true_run(values):
    best = 0
    current = 0

    for value in values:
        if value:
            current += 1
            best = max(best, current)
        else:
            current = 0

    return best


def violation_events(values):
    values = np.asarray(values, dtype=bool)

    if len(values) == 0:
        return 0

    return int(values[0]) + int(np.sum(values[1:] & ~values[:-1]))


def overall_episode_violation_arrays(data):
    max_runs = []
    events = []
    violated = []

    for ep in np.unique(data["episode"]):
        mask = data["episode"] == ep
        flags = np.any(data["joint_speeds"][mask] > joint_thresholds, axis=1)

        max_runs.append(longest_true_run(flags))
        events.append(violation_events(flags))
        violated.append(bool(np.any(flags)))

    return (
        np.asarray(max_runs, dtype=np.int64),
        np.asarray(events, dtype=np.int64),
        np.asarray(violated, dtype=bool),
    )


def per_joint_episode_violation_frequency(data):
    result = []

    for joint_index in range(6):
        violated = []

        for ep in np.unique(data["episode"]):
            mask = data["episode"] == ep
            flags = data["joint_speeds"][mask, joint_index] > joint_thresholds[joint_index]
            violated.append(bool(np.any(flags)))

        result.append(float(np.mean(violated)))

    return np.asarray(result, dtype=np.float64)

In [ ]:
def summarize_policy(data):
    speeds = data["joint_speeds"]
    flags = speeds > joint_thresholds
    any_flags = np.any(flags, axis=1)

    max_runs, events, violated = overall_episode_violation_arrays(data)

    return {
        "mean_episode_reward": float(np.mean(data["episode_rewards"])),
        "std_episode_reward": float(np.std(data["episode_rewards"])),
        "median_episode_reward": float(np.median(data["episode_rewards"])),
        "mean_episode_total_cost": float(np.mean(data["episode_costs"])),
        "median_episode_total_cost": float(np.median(data["episode_costs"])),
        "maximum_one_step_cost": float(np.max(data["cost"])),
        "overall_step_violation_rate": float(np.mean(any_flags)),
        "overall_episode_violation_frequency": float(np.mean(violated)),
        "longest_violation_run": int(np.max(max_runs)),
        "mean_longest_violation_run": float(np.mean(max_runs)),
        "mean_violation_events_per_episode": float(np.mean(events)),
        "mean_maximum_joint_speed": float(np.mean(data["maximum_joint_speed"])),
        "maximum_joint_speed": float(np.max(data["maximum_joint_speed"])),
        "mean_net_displacement": float(np.mean(data["episode_net_displacements"])),
        "mean_absolute_displacement": float(np.mean(data["episode_absolute_displacements"])),
        "mean_traveled_distance": float(np.mean(data["episode_traveled_distances"])),
        "mean_maximum_action_abs": float(np.mean(data["maximum_action_abs"])),
    }


unconstrained_summary = summarize_policy(unconstrained)
joint_summary = summarize_policy(joint_constrained)

summary = pd.DataFrame(
    {
        "Unconstrained": unconstrained_summary,
        "Joint-speed constrained": joint_summary,
    }
)

display(summary)

In [ ]:
comparison = summary.copy()

reference = comparison["Unconstrained"].astype(float)
candidate = comparison["Joint-speed constrained"].astype(float)

comparison["Absolute change"] = candidate - reference
comparison["Percent change"] = np.where(
    np.abs(reference) > 1e-12,
    100.0 * (candidate - reference) / np.abs(reference),
    np.nan,
)

comparison.to_csv(OUTPUT / "summary.csv")
display(comparison)

print("Saved:", OUTPUT / "summary.csv")

In [ ]:
def reduction(before, after):
    if abs(before) <= 1e-12:
        return np.nan
    return 100.0 * (before - after) / abs(before)


reward_retention = (
    100.0
    * joint_summary["mean_episode_reward"]
    / unconstrained_summary["mean_episode_reward"]
)

step_violation_reduction = reduction(
    unconstrained_summary["overall_step_violation_rate"],
    joint_summary["overall_step_violation_rate"],
)

episode_violation_reduction = reduction(
    unconstrained_summary["overall_episode_violation_frequency"],
    joint_summary["overall_episode_violation_frequency"],
)

total_cost_reduction = reduction(
    unconstrained_summary["mean_episode_total_cost"],
    joint_summary["mean_episode_total_cost"],
)

max_cost_reduction = reduction(
    unconstrained_summary["maximum_one_step_cost"],
    joint_summary["maximum_one_step_cost"],
)

print(f"Reward retention: {reward_retention:.2f}%")
print(f"Overall step violation reduction: {step_violation_reduction:.2f}%")
print(f"Overall episode violation-frequency reduction: {episode_violation_reduction:.2f}%")
print(f"Mean episode joint-speed-cost reduction: {total_cost_reduction:.2f}%")
print(f"Maximum one-step cost reduction: {max_cost_reduction:.2f}%")

In [ ]:
def per_joint_statistics(data):
    speeds = data["joint_speeds"]
    step_rates = np.mean(speeds > joint_thresholds, axis=0)
    episode_rates = per_joint_episode_violation_frequency(data)

    rows = []

    for i, name in enumerate(JOINT_NAMES):
        rows.append(
            {
                "joint": name,
                "threshold": joint_thresholds[i],
                "mean": float(np.mean(speeds[:, i])),
                "median": float(np.median(speeds[:, i])),
                "p95": float(np.percentile(speeds[:, i], 95)),
                "p99": float(np.percentile(speeds[:, i], 99)),
                "maximum": float(np.max(speeds[:, i])),
                "step_violation_rate": float(step_rates[i]),
                "episode_violation_frequency": float(episode_rates[i]),
            }
        )

    return pd.DataFrame(rows)


unconstrained_joint_stats = per_joint_statistics(unconstrained)
constrained_joint_stats = per_joint_statistics(joint_constrained)

unconstrained_joint_stats.insert(1, "policy", "Unconstrained")
constrained_joint_stats.insert(1, "policy", "Joint-speed constrained")

joint_stats = pd.concat(
    [unconstrained_joint_stats, constrained_joint_stats],
    ignore_index=True,
)

joint_stats.to_csv(OUTPUT / "per_joint_statistics.csv", index=False)
display(joint_stats)

print("Saved:", OUTPUT / "per_joint_statistics.csv")

In [ ]:
u = unconstrained_joint_stats.set_index("joint")
c = constrained_joint_stats.set_index("joint")

per_joint_comparison = pd.DataFrame(
    {
        "threshold": u["threshold"],
        "unconstrained_step_violation_rate": u["step_violation_rate"],
        "constrained_step_violation_rate": c["step_violation_rate"],
        "unconstrained_episode_violation_frequency": u["episode_violation_frequency"],
        "constrained_episode_violation_frequency": c["episode_violation_frequency"],
        "unconstrained_p99": u["p99"],
        "constrained_p99": c["p99"],
        "unconstrained_maximum": u["maximum"],
        "constrained_maximum": c["maximum"],
    }
)

per_joint_comparison["step_violation_reduction_percent"] = np.where(
    per_joint_comparison["unconstrained_step_violation_rate"] > 0,
    100.0
    * (
        per_joint_comparison["unconstrained_step_violation_rate"]
        - per_joint_comparison["constrained_step_violation_rate"]
    )
    / per_joint_comparison["unconstrained_step_violation_rate"],
    np.nan,
)

per_joint_comparison["p99_reduction_percent"] = np.where(
    per_joint_comparison["unconstrained_p99"] > 0,
    100.0
    * (
        per_joint_comparison["unconstrained_p99"]
        - per_joint_comparison["constrained_p99"]
    )
    / per_joint_comparison["unconstrained_p99"],
    np.nan,
)

per_joint_comparison.to_csv(OUTPUT / "per_joint_comparison.csv")
display(per_joint_comparison)

print("Saved:", OUTPUT / "per_joint_comparison.csv")

In [ ]:
u_runs, u_events, u_violated = overall_episode_violation_arrays(unconstrained)
j_runs, j_events, j_violated = overall_episode_violation_arrays(joint_constrained)

episode_table = pd.DataFrame(
    {
        "seed": np.arange(FIRST_SEED, FIRST_SEED + EPISODES),
        "unconstrained_reward": unconstrained["episode_rewards"],
        "joint_constrained_reward": joint_constrained["episode_rewards"],
        "unconstrained_total_cost": unconstrained["episode_costs"],
        "joint_constrained_total_cost": joint_constrained["episode_costs"],
        "unconstrained_max_cost": unconstrained["episode_max_costs"],
        "joint_constrained_max_cost": joint_constrained["episode_max_costs"],
        "unconstrained_net_displacement": unconstrained["episode_net_displacements"],
        "joint_constrained_net_displacement": joint_constrained["episode_net_displacements"],
        "unconstrained_longest_violation_run": u_runs,
        "joint_constrained_longest_violation_run": j_runs,
        "unconstrained_violation_events": u_events,
        "joint_constrained_violation_events": j_events,
        "unconstrained_any_violation": u_violated,
        "joint_constrained_any_violation": j_violated,
    }
)

episode_table.to_csv(OUTPUT / "episode_comparison.csv", index=False)
display(episode_table)

print("Saved:", OUTPUT / "episode_comparison.csv")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    episode_table["seed"],
    episode_table["unconstrained_reward"],
    label="Unconstrained",
)
plt.plot(
    episode_table["seed"],
    episode_table["joint_constrained_reward"],
    label="Joint-speed constrained",
)
plt.xlabel("Held-out reset seed")
plt.ylabel("Episode reward")
plt.title("Episode reward on identical held-out seeds")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "episode_reward_comparison.png", dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(
    [
        unconstrained["episode_rewards"],
        joint_constrained["episode_rewards"],
    ],
    tick_labels=["Unconstrained", "Joint-speed constrained"],
)
plt.ylabel("Episode reward")
plt.title("Held-out episode reward distribution")
plt.tight_layout()
plt.savefig(OUTPUT / "reward_boxplot.png", dpi=160)
plt.show()

In [ ]:
for i, name in enumerate(JOINT_NAMES):
    low = min(
        float(np.min(unconstrained["joint_speeds"][:, i])),
        float(np.min(joint_constrained["joint_speeds"][:, i])),
    )
    high = max(
        float(np.max(unconstrained["joint_speeds"][:, i])),
        float(np.max(joint_constrained["joint_speeds"][:, i])),
    )

    bins = np.linspace(low, high, 100)

    plt.figure(figsize=(10, 5))
    plt.hist(
        unconstrained["joint_speeds"][:, i],
        bins=bins,
        density=True,
        alpha=0.5,
        label="Unconstrained",
    )
    plt.hist(
        joint_constrained["joint_speeds"][:, i],
        bins=bins,
        density=True,
        alpha=0.5,
        label="Joint-speed constrained",
    )
    plt.axvline(
        joint_thresholds[i],
        linestyle="--",
        label="Threshold",
    )
    plt.xlabel("Absolute joint speed")
    plt.ylabel("Density")
    plt.title(name)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / f"{name}_speed_distribution.png", dpi=160)
    plt.show()

In [ ]:
x = np.arange(6)
width = 0.36

u_rates = unconstrained_joint_stats["step_violation_rate"].to_numpy()
c_rates = constrained_joint_stats["step_violation_rate"].to_numpy()

plt.figure(figsize=(11, 5))
plt.bar(x - width / 2, u_rates, width, label="Unconstrained")
plt.bar(x + width / 2, c_rates, width, label="Joint-speed constrained")
plt.xticks(x, JOINT_NAMES, rotation=25)
plt.ylabel("Step violation rate")
plt.title("Per-joint threshold violation rate")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "per_joint_step_violation_rates.png", dpi=160)
plt.show()

In [ ]:
x = np.arange(6)
width = 0.36

u_p99 = unconstrained_joint_stats["p99"].to_numpy()
c_p99 = constrained_joint_stats["p99"].to_numpy()

plt.figure(figsize=(11, 5))
plt.bar(x - width / 2, u_p99, width, label="Unconstrained")
plt.bar(x + width / 2, c_p99, width, label="Joint-speed constrained")
plt.scatter(x, joint_thresholds, marker="_", s=300, label="Threshold")
plt.xticks(x, JOINT_NAMES, rotation=25)
plt.ylabel("Absolute joint speed")
plt.title("Per-joint P99 speed")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "per_joint_p99.png", dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    episode_table["seed"],
    episode_table["unconstrained_total_cost"],
    label="Unconstrained",
)
plt.plot(
    episode_table["seed"],
    episode_table["joint_constrained_total_cost"],
    label="Joint-speed constrained",
)
plt.xlabel("Held-out reset seed")
plt.ylabel("Episode total joint-speed cost")
plt.title("Joint-speed cost on identical held-out seeds")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "episode_total_cost_comparison.png", dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    episode_table["seed"],
    episode_table["unconstrained_longest_violation_run"],
    label="Unconstrained",
)
plt.plot(
    episode_table["seed"],
    episode_table["joint_constrained_longest_violation_run"],
    label="Joint-speed constrained",
)
plt.xlabel("Held-out reset seed")
plt.ylabel("Longest consecutive violation run")
plt.title("Longest any-joint violation per episode")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "violation_run_comparison.png", dpi=160)
plt.show()

In [ ]:
u_fast = np.bincount(
    unconstrained["fastest_joint_index"],
    minlength=6,
).astype(np.float64)

c_fast = np.bincount(
    joint_constrained["fastest_joint_index"],
    minlength=6,
).astype(np.float64)

u_fast /= np.sum(u_fast)
c_fast /= np.sum(c_fast)

x = np.arange(6)
width = 0.36

plt.figure(figsize=(11, 5))
plt.bar(x - width / 2, u_fast, width, label="Unconstrained")
plt.bar(x + width / 2, c_fast, width, label="Joint-speed constrained")
plt.xticks(x, JOINT_NAMES, rotation=25)
plt.ylabel("Fraction of timesteps")
plt.title("Which joint is fastest at each timestep")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "fastest_joint_frequency.png", dpi=160)
plt.show()

In [ ]:
def worst_episode_by_cost(data):
    index = int(np.argmax(data["episode_costs"]))
    episode_number = index + 1
    seed = int(data["reset_seed"][data["episode"] == episode_number][0])
    return episode_number, seed


u_worst_episode, u_worst_seed = worst_episode_by_cost(unconstrained)
j_worst_episode, j_worst_seed = worst_episode_by_cost(joint_constrained)

print("Unconstrained worst cost episode:", u_worst_episode, "seed:", u_worst_seed)
print("Joint-speed constrained worst cost episode:", j_worst_episode, "seed:", j_worst_seed)

In [ ]:
mask = unconstrained["episode"] == u_worst_episode

plt.figure(figsize=(12, 6))

for i, name in enumerate(JOINT_NAMES):
    plt.plot(
        unconstrained["step"][mask],
        unconstrained["joint_speeds"][mask, i] / joint_thresholds[i],
        label=name,
    )

plt.axhline(1.0, linestyle="--", label="Threshold")
plt.xlabel("RL step")
plt.ylabel("Joint speed / threshold")
plt.title(f"Unconstrained worst joint-speed-cost episode, seed {u_worst_seed}")
plt.legend(ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT / "unconstrained_worst_joint_speed_episode.png", dpi=160)
plt.show()

In [ ]:
mask = joint_constrained["episode"] == j_worst_episode

plt.figure(figsize=(12, 6))

for i, name in enumerate(JOINT_NAMES):
    plt.plot(
        joint_constrained["step"][mask],
        joint_constrained["joint_speeds"][mask, i] / joint_thresholds[i],
        label=name,
    )

plt.axhline(1.0, linestyle="--", label="Threshold")
plt.xlabel("RL step")
plt.ylabel("Joint speed / threshold")
plt.title(f"Joint-speed constrained worst cost episode, seed {j_worst_seed}")
plt.legend(ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT / "joint_constrained_worst_joint_speed_episode.png", dpi=160)
plt.show()

In [ ]:
TRAINING_DATA = PROJECT / "data_train" / "HalfCheetahJointSpeedCost" / f"run{RUN}"

reward_history_path = TRAINING_DATA / "evaluate_rewards.npy"
cost_history_path = TRAINING_DATA / "evaluate_costs.npy"
max_cost_history_path = TRAINING_DATA / "evaluate_max_costs.npy"

training_histories_exist = (
    reward_history_path.exists()
    and cost_history_path.exists()
    and max_cost_history_path.exists()
)

print("Training histories found:", training_histories_exist)
print("Training data directory:", TRAINING_DATA)

In [ ]:
if training_histories_exist:
    evaluation_rewards = np.load(reward_history_path)
    evaluation_costs = np.load(cost_history_path)
    evaluation_max_costs = np.load(max_cost_history_path)
    evaluation_episode = np.arange(len(evaluation_rewards)) * EVALUATE_FREQ

    print("Evaluation checkpoints:", len(evaluation_rewards))
    print("Last five rewards:", evaluation_rewards[-5:])
    print("Last five total costs:", evaluation_costs[-5:])
    print("Last five maximum costs:", evaluation_max_costs[-5:])

In [ ]:
if training_histories_exist:
    plt.figure(figsize=(10, 5))
    plt.plot(evaluation_episode, evaluation_rewards)
    plt.xlabel("Training episode")
    plt.ylabel("Evaluation reward")
    plt.title("Joint-speed-constrained training evaluation reward")
    plt.tight_layout()
    plt.savefig(OUTPUT / "training_evaluation_reward.png", dpi=160)
    plt.show()

In [ ]:
if training_histories_exist:
    plt.figure(figsize=(10, 5))
    plt.plot(evaluation_episode, evaluation_costs)
    plt.xlabel("Training episode")
    plt.ylabel("Evaluation total joint-speed cost")
    plt.title("Joint-speed-constrained training evaluation cost")
    plt.tight_layout()
    plt.savefig(OUTPUT / "training_evaluation_cost.png", dpi=160)
    plt.show()

In [ ]:
if training_histories_exist:
    plt.figure(figsize=(10, 5))
    plt.plot(
        evaluation_episode,
        evaluation_max_costs,
        label="Evaluation max cost",
    )
    plt.axhline(
        0.1,
        linestyle="--",
        label="persistent_eps = 0.1",
    )
    plt.xlabel("Training episode")
    plt.ylabel("Maximum one-step joint-speed cost")
    plt.title("Joint-speed-constrained maximum evaluation cost")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / "training_evaluation_max_cost.png", dpi=160)
    plt.show()

In [ ]:
final_result = pd.DataFrame(
    [
        {
            "metric": "Mean episode reward",
            "unconstrained": unconstrained_summary["mean_episode_reward"],
            "joint_speed_constrained": joint_summary["mean_episode_reward"],
        },
        {
            "metric": "Mean episode total joint-speed cost",
            "unconstrained": unconstrained_summary["mean_episode_total_cost"],
            "joint_speed_constrained": joint_summary["mean_episode_total_cost"],
        },
        {
            "metric": "Maximum one-step joint-speed cost",
            "unconstrained": unconstrained_summary["maximum_one_step_cost"],
            "joint_speed_constrained": joint_summary["maximum_one_step_cost"],
        },
        {
            "metric": "Overall step violation rate",
            "unconstrained": unconstrained_summary["overall_step_violation_rate"],
            "joint_speed_constrained": joint_summary["overall_step_violation_rate"],
        },
        {
            "metric": "Overall episode violation frequency",
            "unconstrained": unconstrained_summary["overall_episode_violation_frequency"],
            "joint_speed_constrained": joint_summary["overall_episode_violation_frequency"],
        },
        {
            "metric": "Longest violation run",
            "unconstrained": unconstrained_summary["longest_violation_run"],
            "joint_speed_constrained": joint_summary["longest_violation_run"],
        },
        {
            "metric": "Maximum joint speed",
            "unconstrained": unconstrained_summary["maximum_joint_speed"],
            "joint_speed_constrained": joint_summary["maximum_joint_speed"],
        },
        {
            "metric": "Mean net displacement",
            "unconstrained": unconstrained_summary["mean_net_displacement"],
            "joint_speed_constrained": joint_summary["mean_net_displacement"],
        },
    ]
)

final_result["absolute_change"] = (
    final_result["joint_speed_constrained"]
    - final_result["unconstrained"]
)

final_result["percent_change"] = np.where(
    np.abs(final_result["unconstrained"]) > 1e-12,
    100.0
    * final_result["absolute_change"]
    / np.abs(final_result["unconstrained"]),
    np.nan,
)

final_result.to_csv(
    OUTPUT / "final_heldout_result.csv",
    index=False,
)

display(final_result)

print()
print(f"Reward retention: {reward_retention:.2f}%")
print(f"Overall step violation reduction: {step_violation_reduction:.2f}%")
print(f"Overall episode violation-frequency reduction: {episode_violation_reduction:.2f}%")
print(f"Mean episode joint-speed-cost reduction: {total_cost_reduction:.2f}%")
print(f"Maximum one-step cost reduction: {max_cost_reduction:.2f}%")
print("Output directory:", OUTPUT)